In [1]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim import Adam
from torchvision import datasets, transforms


In [2]:
import random
import torch
import torch.nn as nn
from torchvision.utils import make_grid
import torch.optim as optim
import numpy as np
import torch.utils.data
import torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.utils as vutils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
class Generator(nn.Module):

    def __init__(self):
        super(Generator, self).__init__()

        # input 100*1*1
        self.layer1 = nn.Sequential(nn.ConvTranspose2d(100, 512, 4, 1, 0, bias=False),
                                    nn.BatchNorm2d(512),
                                    nn.ReLU(True))

        # input 512*4*4
        self.layer2 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(256),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 256*8*8
        self.layer3 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(128),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 128*16*16
        self.layer4 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(64),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 64*32*32
        self.layer5 = nn.Sequential(nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
                                    nn.Tanh())

        # output 1*64*64

        self.embedding = nn.Embedding(10, 100)

    def forward(self, noise, label):  # noise shape: (,100)

        label_embedding = self.embedding(label)
        x = torch.mul(noise, label_embedding)
        x = x.view(-1, 100, 1, 1)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return x


class Discriminator(nn.Module):

    def __init__(self):
        super(Discriminator, self).__init__()

        # input 1*64*64
        self.layer1 = nn.Sequential(nn.Conv2d(1, 64, 4, 2, 1, bias=False),
                                    nn.LeakyReLU(0.2, True))

        # input 64*32*32
        self.layer2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(128),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.7))
        # input 128*16*16
        self.layer3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(256),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.6))
        # input 256*8*8
        self.layer4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(512),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.5))
        # input 512*4*4
        self.validity_layer = nn.Sequential(nn.Conv2d(512, 1, 4, 1, 0, bias=False),
                                            nn.Sigmoid())

        self.label_layer = nn.Sequential(nn.Conv2d(512, 10, 4, 1, 0, bias=False),
                                         nn.LogSoftmax(dim=1))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        validity = self.validity_layer(x)
        plabel = self.label_layer(x)

        validity = validity.view(-1)
        plabel = plabel.view(-1, 10)

        return validity, plabel

In [4]:

import os
bestEpoch=2


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)


gen = Generator().to(device)

disc = Discriminator().to(device)


paramsG = list(gen.parameters())
print(len(paramsG))

paramsD = list(disc.parameters())
print(len(paramsD))

optimG = optim.Adam(gen.parameters(), 0.0002, betas=(0.5, 0.999))   
optimD = optim.Adam(disc.parameters(), 0.0002, betas=(0.5, 0.999))

# label smoothening
real_labels = 0.7 + 0.5 * torch.rand(7, device=device)   # 0.7 - 1.2
fake_labels = 0.3 * torch.rand(7, device=device)   # 0 - 0.3
epochs = 1

validity_loss = nn.BCELoss()
save_dir="./res1030/dcgan2nsl"



gen.load_state_dict(torch.load('./res1030/dcgan2nsl/dcgan-2nsl-epoch-2.GNet'))

disc.load_state_dict(torch.load('./res1030/dcgan2nsl/dcgan-2nsl-epoch-2.DNet'))

14
12


<All keys matched successfully>

In [5]:
yuzhi1=7000

In [6]:
j=0
noise = torch.randn(yuzhi1, 100, device=device)
labels = [j for i in range(yuzhi1)]
labels=torch.tensor(labels).to(device)
fakes = gen(noise, labels).detach()

lab=np.array(labels.cpu())
fak=np.array(fakes.cpu())

In [7]:
fak.shape

(7000, 1, 64, 64)

In [8]:
fakes.shape

torch.Size([7000, 1, 64, 64])

In [9]:
for j in range(7):
    if j+1==4:
        continue
    noise = torch.randn(yuzhi1, 100, device=device)
    labelszy = [j+1 for i in range(yuzhi1)]
    labelszy=torch.tensor(labelszy).to(device)
    fakeszy = gen(noise, labelszy).detach()
    labelszyy=labelszy.cpu()
    fakeszyy=fakeszy.cpu()
    
    fak = np.concatenate((fak, fakeszyy)) 
    lab = np.concatenate((lab, labelszyy)) 
    


In [10]:
yuzhizy3=7000

noise = torch.randn(yuzhizy3, 100, device=device)
labelszy = [0 for i in range(yuzhizy3)]
labelszy=torch.tensor(labelszy).to(device)
fakeszy = gen(noise, labelszy).detach()

labelszyy=labelszy.cpu()
fakeszyy=fakeszy.cpu()

fak = np.concatenate((fak, fakeszyy)) 
lab = np.concatenate((lab, labelszyy)) 

In [11]:
noise = torch.randn(yuzhizy3, 100, device=device)
labelszy = [6 for i in range(yuzhizy3)]
labelszy=torch.tensor(labelszy).to(device)
fakeszy = gen(noise, labelszy).detach()
labelszyy=labelszy.cpu()
fakeszyy=fakeszy.cpu()

fak = np.concatenate((fak, fakeszyy)) 
lab = np.concatenate((lab, labelszyy)) 

In [12]:
noise = torch.randn(yuzhizy3, 100, device=device)
labelszy = [7 for i in range(yuzhizy3)]
labelszy=torch.tensor(labelszy).to(device)
fakeszy = gen(noise, labelszy).detach()
labelszyy=labelszy.cpu()
fakeszyy=fakeszy.cpu()

fak = np.concatenate((fak, fakeszyy)) 
lab = np.concatenate((lab, labelszyy)) 

In [13]:
fak.shape

(70000, 1, 64, 64)

In [14]:
unique,counts = np.unique(lab,return_counts=True)
print(unique, counts)

[0 1 2 3 5 6 7] [14000  7000  7000  7000  7000 14000 14000]


In [15]:
np.save("./data/acgan-iot-data-f.npy",fak)
print("fak",fak.shape)

fak (70000, 1, 64, 64)


In [16]:
np.save("./data/acgan-iot-label-f.npy",lab)
print("lab",lab.shape)

lab (70000,)
